# Phase 2: Experiment Tracking & Model Registry with MLflow

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("steam.csv")
def has_term(val, target):
    if not isinstance(val, str): return 0
    return 1 if any(t.strip().lower() == target.lower() for t in val.split(';')) else 0
df['release_year'] = pd.to_datetime(df['release_date'], errors='coerce').dt.year
df['release_year'] = df['release_year'].fillna(df['release_year'].median())
df['self_published'] = (df['developer'] == df['publisher']).astype(int)
df['is_mac'] = df['platforms'].apply(lambda x: 1 if isinstance(x, str) and 'mac' in x.lower() else 0)
df['is_multiplayer'] = df['categories'].apply(lambda x: has_term(x, 'Multi-player'))
df['is_indie'] = df['genres'].apply(lambda x: has_term(x, 'Indie'))
df['is_action'] = df['genres'].apply(lambda x: has_term(x, 'Action'))
df['is_early_access'] = df['genres'].apply(lambda x: has_term(x, 'Early Access'))
features = ['average_playtime', 'achievements', 'release_year', 'self_published', 'english', 'is_mac', 'is_multiplayer', 'is_indie', 'is_action', 'is_early_access']
target = 'price'
df_clean = df.dropna(subset=features + [target])
df_clean = df_clean[(df_clean['price'] >= 4.0) & (df_clean['price'] <= 80.0)]
df_clean = df_clean[~((df_clean['price'] <= 6.0) & (df_clean['achievements'] > 50))]

X = df_clean[features]; y = df_clean[target]
df_clean['price_tier'] = pd.cut(df_clean['price'], bins=[0, 15, 40, 100], labels=[1, 2, 3])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=df_clean['price_tier'])

In [ ]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("SteamGamesPricePrediction_v1")

In [ ]:
with mlflow.start_run() as run:
    mlflow.set_tag("algorithm", "GradientBoostingRegressor")
    mlflow.set_tag("dataset_version", "steam_v1")
    model = GradientBoostingRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    mlflow.log_metric("MAE", mean_absolute_error(y_test, model.predict(X_test)))
    mlflow.log_metric("RMSE", np.sqrt(mean_squared_error(y_test, model.predict(X_test))))
    mlflow.log_metric("R2", r2_score(y_test, model.predict(X_test)))
    mlflow.sklearn.log_model(model, "model")
    mlflow.register_model(f"runs:/{run.info.run_id}/model", "BestRegressor_v1")

In [ ]:
alpha = 0.05
lr_sm = sm.OLS(y_train, sm.add_constant(X_train)).fit()
conf_interval = lr_sm.conf_int(alpha)
print(conf_interval)

## Interpretation of the Confidence Intervals

The 95% confidence intervals give the range within which the true regression coefficient lies with 95% certainty. A coefficient interval entirely above zero means the feature has a statistically significant positive effect on price. An interval that crosses zero suggests the feature may not reliably influence the prediction.